# TalentCLEF Task B — Phase 0 & Phase 1: Val + Test Evaluation
## Using Official TalentCLEF Evaluation Script (ranx)

This notebook runs all 39 experiments on **both** the validation and test sets,
evaluates using the official `talentclef_evaluate.py` script (ranx library),
and logs all results to W&B with `val_` and `test_` prefixes.

**Official Metrics:** MAP, MRR, NDCG, P@5, P@10, P@100

---
## 0. Setup

In [ ]:
# ============================================================
# Install dependencies
# ============================================================
!pip install -q rank_bm25 sentence-transformers transformers torch
!pip install -q openai anthropic cohere voyageai
!pip install -q wandb pandas numpy tqdm matplotlib seaborn
!pip install -q FlagEmbedding accelerate bitsandbytes
!pip install -q ranx  # Official evaluation library

In [ ]:
# ============================================================
# Imports
# ============================================================
import os, json, ast, time
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from collections import defaultdict
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from ranx import Qrels, Run, evaluate as ranx_evaluate

import openai, anthropic, cohere, voyageai
import wandb

### Upload `config_api.py`
Upload your `config_api.py` before running the next cell.

In [ ]:
# ============================================================
# Load configuration
# ============================================================
import config_api

CONFIG = {
    "OPENAI_API_KEY": config_api.OPENAI_API_KEY,
    "ANTHROPIC_API_KEY": config_api.ANTHROPIC_API_KEY,
    "COHERE_API_KEY": config_api.COHERE_API_KEY,
    "VOYAGE_API_KEY": config_api.VOYAGE_API_KEY,
    "WANDB_API_KEY": config_api.WANDB_API_KEY,
    "TOGETHER_API_KEY": config_api.TOGETHER_API_KEY,
    "LLM_QWEN": config_api.LLM_QWEN,
    "LLM_OPUS": config_api.LLM_OPUS,
    "LLM_GPT": config_api.LLM_GPT,
    "QWEN_BASE_URL": config_api.QWEN_BASE_URL,
    "BIENC_QWEN": config_api.BIENC_QWEN,
    "BIENC_BGE": config_api.BIENC_BGE,
    "BIENC_OPENAI": config_api.BIENC_OPENAI,
    "CROSSENC_BGE": config_api.CROSSENC_BGE,
    "CROSSENC_QWEN": config_api.CROSSENC_QWEN,
    "CROSSENC_VOYAGE": config_api.CROSSENC_VOYAGE,
    "CROSSENC_COHERE": config_api.CROSSENC_COHERE,
    "RERANK_TOP_K": config_api.RERANK_TOP_K,
    "RRF_K": config_api.RRF_K,
    "WANDB_PROJECT": config_api.WANDB_PROJECT,
    "DATA_DIR": config_api.DATA_DIR,
    "OUTPUT_DIR": config_api.OUTPUT_DIR,
    "DESC_DIR": config_api.DESC_DIR,
}

os.environ["OPENAI_API_KEY"] = CONFIG["OPENAI_API_KEY"]
os.environ["ANTHROPIC_API_KEY"] = CONFIG["ANTHROPIC_API_KEY"]
os.environ["COHERE_API_KEY"] = CONFIG["COHERE_API_KEY"]
os.environ["VOYAGE_API_KEY"] = CONFIG["VOYAGE_API_KEY"]

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
os.makedirs(CONFIG["DESC_DIR"], exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Rate limits
COHERE_SLEEP = 6
VOYAGE_SLEEP = 2

In [ ]:
# ============================================================
# W&B Login
# ============================================================
if CONFIG["WANDB_API_KEY"]:
    wandb.login(key=CONFIG["WANDB_API_KEY"])
    print("W&B logged in")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("W&B offline mode")

---
## 1. Load Data (Val + Test)

In [ ]:
# ============================================================
# Download data
# ============================================================
if not os.path.exists(CONFIG["DATA_DIR"]):
    !wget -q https://zenodo.org/records/14693201/files/TaskB.zip
    !unzip -q TaskB.zip -d taskB
    print("Data downloaded")
else:
    print("Data already exists")

In [ ]:
# ============================================================
# Load all data
# ============================================================
train_path = os.path.join(CONFIG["DATA_DIR"], "training")
dev_path = os.path.join(CONFIG["DATA_DIR"], "validation")
test_path = os.path.join(CONFIG["DATA_DIR"], "test")

# Training (for reference)
job2skill = pd.read_csv(os.path.join(train_path, "job2skill.tsv"), sep="\t",
                        names=["job_id", "skill_id", "rel_type"])
with open(os.path.join(train_path, "jobid2terms.json")) as f: jobid2terms = json.load(f)
with open(os.path.join(train_path, "skillid2terms.json")) as f: skillid2terms = json.load(f)

# Dev/Val
dev_queries_df = pd.read_csv(os.path.join(dev_path, "queries"), sep="\t")
dev_corpus_df = pd.read_csv(os.path.join(dev_path, "corpus_elements"), sep="\t")
dev_qrels_df = pd.read_csv(os.path.join(dev_path, "qrels.tsv"), sep="\t",
                            names=["q_id", "iter", "c_id", "relevance"])

# Test
test_queries_df = pd.read_csv(os.path.join(test_path, "queries"), sep="\t")
test_corpus_df = pd.read_csv(os.path.join(test_path, "corpus_elements"), sep="\t")

# Check if test qrels exist
test_qrels_path = os.path.join(test_path, "qrels.tsv")
has_test_qrels = os.path.exists(test_qrels_path)
if has_test_qrels:
    test_qrels_df = pd.read_csv(test_qrels_path, sep="\t",
                                names=["q_id", "iter", "c_id", "relevance"])
    print(f"Test qrels found: {len(test_qrels_df)} entries")
else:
    # Also try .txt extension
    test_qrels_path2 = os.path.join(test_path, "qrels.txt")
    if os.path.exists(test_qrels_path2):
        test_qrels_df = pd.read_csv(test_qrels_path2, sep="\t",
                                    names=["q_id", "iter", "c_id", "relevance"])
        has_test_qrels = True
        print(f"Test qrels found (.txt): {len(test_qrels_df)} entries")
    else:
        test_qrels_df = None
        print("No test qrels found — test runs will generate TREC files only (no evaluation)")

print(f"\nVal: {len(dev_queries_df)} queries, {len(dev_corpus_df)} corpus skills")
print(f"Test: {len(test_queries_df)} queries, {len(test_corpus_df)} corpus skills")

---
## 2. Data Preparation

In [ ]:
# ============================================================
# Prepare data into standardized format
# ============================================================

def prepare_split(queries_df, corpus_df, qrels_df=None):
    """Prepare a data split into standardized dicts."""
    queries = dict(zip(queries_df.iloc[:, 0], queries_df.iloc[:, 1]))

    corpus = {}
    for _, row in corpus_df.iterrows():
        aliases = row['skill_aliases']
        if isinstance(aliases, str):
            try: aliases = ast.literal_eval(aliases)
            except: aliases = [aliases]
        corpus[row['c_id']] = {"esco_uri": row.get('esco_uri', ''), "aliases": aliases}

    qrels = None
    if qrels_df is not None:
        qrels = defaultdict(set)
        for _, row in qrels_df.iterrows():
            if row['relevance'] == 1:
                qrels[row.iloc[0]].add(row.iloc[2])
        qrels = dict(qrels)

    return queries, corpus, qrels

# Prepare both splits
val_q, val_c, val_qr = prepare_split(dev_queries_df, dev_corpus_df, dev_qrels_df)
test_q, test_c, test_qr = prepare_split(test_queries_df, test_corpus_df,
                                          test_qrels_df if has_test_qrels else None)

DATASETS = {
    "val": {"queries": val_q, "corpus": val_c, "qrels": val_qr, "has_qrels": True},
    "test": {"queries": test_q, "corpus": test_c, "qrels": test_qr, "has_qrels": has_test_qrels},
}

for name, ds in DATASETS.items():
    print(f"{name}: {len(ds['queries'])} queries, {len(ds['corpus'])} corpus, qrels={'yes' if ds['has_qrels'] else 'no'}")

---
## 3. Utility Functions

In [ ]:
# ============================================================
# Text helpers
# ============================================================

def get_corpus_text(corpus, c_id, descriptions=None):
    aliases = corpus[c_id]["aliases"]
    text = ", ".join(aliases) if isinstance(aliases, list) else str(aliases)
    if descriptions and c_id in descriptions:
        text = f"Skill: {text} [SEP] Description: {descriptions[c_id]}"
    return text

def get_query_text(query_title, descriptions=None, q_id=None):
    text = query_title
    if descriptions and q_id and q_id in descriptions:
        text = f"Job: {query_title} [SEP] Description: {descriptions[q_id]}"
    return text

def results_to_ranked(scores_dict):
    run_results = {}
    for q_id, skill_scores in scores_dict.items():
        run_results[q_id] = sorted(skill_scores.items(), key=lambda x: x[1], reverse=True)
    return run_results

print("Text helpers ready")

In [ ]:
# ============================================================
# Official evaluation using ranx
# ============================================================

OFFICIAL_METRICS = ["map", "mrr", "ndcg", "precision@5", "precision@10", "precision@100"]

def save_trec_run(run_results, run_name, split_name):
    """Save results in TREC run file format."""
    path = os.path.join(CONFIG["OUTPUT_DIR"], f"{split_name}_{run_name}.trec")
    with open(path, 'w') as f:
        for q_id in run_results:
            for rank, (c_id, score) in enumerate(run_results[q_id]):
                f.write(f"{q_id}\tQ0\t{c_id}\t{rank+1}\t{score:.6f}\t{split_name}_{run_name}\n")
    return path

def evaluate_official(run_results, qrels_df, run_name, split_name):
    """Evaluate using the official ranx-based evaluation."""
    # Save TREC file
    trec_path = save_trec_run(run_results, run_name, split_name)

    # Load as ranx objects
    qrels_ranx = Qrels.from_df(
        qrels_df[["q_id", "iter", "c_id", "relevance"]].rename(
            columns={"q_id": "q_id", "c_id": "doc_id", "relevance": "score"}
        ) if "q_id" in qrels_df.columns else qrels_df,
        q_id_col="q_id" if "q_id" in qrels_df.columns else qrels_df.columns[0],
        doc_id_col="doc_id" if "doc_id" in qrels_df.columns else "c_id",
        score_col="score" if "score" in qrels_df.columns else "relevance"
    )

    run_df = pd.read_csv(trec_path, sep="\t", header=None,
                         names=["q_id", "Q0", "doc_id", "rank", "score", "tag"],
                         dtype={"q_id": str, "doc_id": str})
    run_ranx = Run.from_df(run_df, q_id_col="q_id", doc_id_col="doc_id", score_col="score")

    # Evaluate
    results = ranx_evaluate(qrels_ranx, run_ranx, OFFICIAL_METRICS)

    metrics = {}
    for metric, score in results.items():
        metrics[metric] = float(score)

    return metrics, trec_path

print("Official evaluation ready")
print(f"Metrics: {OFFICIAL_METRICS}")

In [ ]:
# ============================================================
# Properly load qrels for ranx evaluation
# ============================================================

def load_qrels_for_ranx(qrels_df):
    """Convert qrels DataFrame to ranx Qrels object."""
    df = qrels_df.copy()
    df.columns = ["q_id", "iter", "doc_id", "rel"]
    df["q_id"] = df["q_id"].astype(str)
    df["doc_id"] = df["doc_id"].astype(str)
    return Qrels.from_df(df, q_id_col="q_id", doc_id_col="doc_id", score_col="rel")

def evaluate_with_ranx(run_results, qrels_df, run_name, split_name):
    """Full evaluation pipeline: save TREC file, evaluate with ranx."""
    trec_path = save_trec_run(run_results, run_name, split_name)

    # Load qrels
    qrels_ranx = load_qrels_for_ranx(qrels_df)

    # Load run
    run_df = pd.read_csv(trec_path, sep="\s+", header=None)
    if run_df.shape[1] == 6:
        run_df.columns = ["q_id", "Q0", "doc_id", "rank", "score", "tag"]
    elif run_df.shape[1] == 5:
        run_df.columns = ["q_id", "Q0", "doc_id", "rank", "score"]
    run_df["q_id"] = run_df["q_id"].astype(str)
    run_df["doc_id"] = run_df["doc_id"].astype(str)
    run_ranx = Run.from_df(run_df, q_id_col="q_id", doc_id_col="doc_id", score_col="score")

    # Evaluate
    results = ranx_evaluate(qrels_ranx, run_ranx, OFFICIAL_METRICS)
    metrics = {k: float(v) for k, v in results.items()}

    return metrics, trec_path

print("ranx evaluation pipeline ready")

In [ ]:
# ============================================================
# W&B logging
# ============================================================

ALL_RESULTS = {"val": [], "test": []}

def log_to_wandb(run_name, split_name, group, metrics, config_extra=None, trec_path=None):
    """Log experiment to W&B with split prefix."""
    wb_name = f"{split_name}_{run_name}"
    wb_group = f"{split_name}_{group}"

    config = {"run_name": run_name, "split": split_name, "group": group}
    if config_extra:
        config.update(config_extra)

    run = wandb.init(
        project=CONFIG["WANDB_PROJECT"],
        name=wb_name,
        group=wb_group,
        config=config,
        reinit=True,
    )
    wandb.log(metrics)

    if trec_path and os.path.exists(trec_path):
        artifact = wandb.Artifact(f"trec-{wb_name}", type="predictions")
        artifact.add_file(trec_path)
        wandb.log_artifact(artifact)

    wandb.finish()

    ALL_RESULTS[split_name].append({"run_name": run_name, "group": group, **metrics})
    m = metrics
    print(f"  [{wb_name}] MAP={m.get('map',0):.4f}  MRR={m.get('mrr',0):.4f}  "
          f"NDCG={m.get('ndcg',0):.4f}  P@5={m.get('precision@5',0):.4f}  "
          f"P@10={m.get('precision@10',0):.4f}  P@100={m.get('precision@100',0):.4f}")

def run_and_log(run_name, group, scores_dict, split_name, qrels_df, config_extra=None):
    """Rank scores, evaluate, save, and log — all in one."""
    ranked = results_to_ranked(scores_dict)

    if qrels_df is not None:
        metrics, trec_path = evaluate_with_ranx(ranked, qrels_df, run_name, split_name)
    else:
        trec_path = save_trec_run(ranked, run_name, split_name)
        metrics = {"map": None, "mrr": None, "ndcg": None,
                   "precision@5": None, "precision@10": None, "precision@100": None}
        print(f"  [{split_name}_{run_name}] No qrels — TREC file saved only: {trec_path}")

    log_to_wandb(run_name, split_name, group, metrics, config_extra, trec_path)
    return ranked

print("W&B logging ready")

---
## 4. Retrieval Functions (Reused from Phase 1)

In [ ]:
# ============================================================
# BM25
# ============================================================

def run_bm25(queries, corpus, query_descriptions=None):
    c_ids = list(corpus.keys())
    corpus_texts = []
    for c_id in c_ids:
        aliases = corpus[c_id]["aliases"]
        text = " ".join(aliases) if isinstance(aliases, list) else str(aliases)
        corpus_texts.append(text.lower().split())

    bm25 = BM25Okapi(corpus_texts)

    scores = {}
    for q_id, jobtitle in tqdm(queries.items(), desc="BM25"):
        query_text = jobtitle
        if query_descriptions and q_id in query_descriptions:
            query_text = f"{jobtitle} {query_descriptions[q_id]}"
        tokenized_query = query_text.lower().split()
        doc_scores = bm25.get_scores(tokenized_query)
        scores[q_id] = {c_ids[i]: float(doc_scores[i]) for i in range(len(c_ids))}

    return scores

print("BM25 ready")

In [ ]:
# ============================================================
# Bi-Encoder (local + OpenAI)
# ============================================================

def run_biencoder_local(queries, corpus, model_name, query_prefix="", doc_prefix="",
                        query_descriptions=None, batch_size=64):
    print(f"  Loading model: {model_name}")
    model = SentenceTransformer(model_name, device=DEVICE, trust_remote_code=True)

    c_ids = list(corpus.keys())
    corpus_texts = [f"{doc_prefix}{get_corpus_text(corpus, c_id)}" if doc_prefix
                    else get_corpus_text(corpus, c_id) for c_id in c_ids]

    q_ids = list(queries.keys())
    query_texts = [f"{query_prefix}{get_query_text(queries[q_id], descriptions=query_descriptions, q_id=q_id)}"
                   if query_prefix else get_query_text(queries[q_id], descriptions=query_descriptions, q_id=q_id)
                   for q_id in q_ids]

    print(f"  Encoding {len(corpus_texts)} corpus + {len(query_texts)} queries...")
    corpus_emb = model.encode(corpus_texts, batch_size=batch_size, show_progress_bar=True,
                               convert_to_numpy=True, normalize_embeddings=True)
    query_emb = model.encode(query_texts, batch_size=batch_size, show_progress_bar=True,
                              convert_to_numpy=True, normalize_embeddings=True)

    sim_matrix = query_emb @ corpus_emb.T
    scores = {q_ids[i]: {c_ids[j]: float(sim_matrix[i, j]) for j in range(len(c_ids))}
              for i in range(len(q_ids))}

    del model, corpus_emb, query_emb
    torch.cuda.empty_cache()
    return scores

def run_biencoder_openai(queries, corpus, model_name=None,
                          query_descriptions=None, batch_size=100):
    model_name = model_name or CONFIG["BIENC_OPENAI"]
    client = openai.OpenAI()

    c_ids = list(corpus.keys())
    corpus_texts = [get_corpus_text(corpus, c_id) for c_id in c_ids]
    q_ids = list(queries.keys())
    query_texts = [get_query_text(queries[q_id], descriptions=query_descriptions, q_id=q_id)
                   for q_id in q_ids]

    def embed_batch(texts, model):
        embeddings = []
        for i in tqdm(range(0, len(texts), batch_size), desc="OpenAI embed"):
            batch = texts[i:i+batch_size]
            response = client.embeddings.create(model=model, input=batch)
            embeddings.extend([d.embedding for d in response.data])
        return np.array(embeddings)

    corpus_emb = embed_batch(corpus_texts, model_name)
    query_emb = embed_batch(query_texts, model_name)

    corpus_emb = corpus_emb / np.linalg.norm(corpus_emb, axis=1, keepdims=True)
    query_emb = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True)
    sim_matrix = query_emb @ corpus_emb.T

    scores = {q_ids[i]: {c_ids[j]: float(sim_matrix[i, j]) for j in range(len(c_ids))}
              for i in range(len(q_ids))}
    return scores

print("Bi-encoder functions ready")

In [ ]:
# ============================================================
# Cross-Encoder (local + Qwen + Cohere + Voyage)
# ============================================================

def run_crossencoder_local(queries, corpus, model_name, query_descriptions=None, batch_size=128):
    print(f"  Loading model: {model_name}")
    model = CrossEncoder(model_name, device=DEVICE, trust_remote_code=True)
    q_ids = list(queries.keys())
    c_ids = list(corpus.keys())

    scores = {}
    for q_id in tqdm(q_ids, desc=f"CrossEnc ({model_name.split('/')[-1]})"):
        query_text = get_query_text(queries[q_id], descriptions=query_descriptions, q_id=q_id)
        pairs = [(query_text, get_corpus_text(corpus, c_id)) for c_id in c_ids]
        pair_scores = model.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        scores[q_id] = {c_ids[i]: float(pair_scores[i]) for i in range(len(c_ids))}

    del model; torch.cuda.empty_cache()
    return scores

def run_crossencoder_qwen_reranker(queries, corpus, model_name, query_descriptions=None, batch_size=16):
    print(f"  Loading Qwen Reranker: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left', trust_remote_code=True)
    model_qwen = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map=DEVICE, trust_remote_code=True)
    model_qwen.eval()

    token_true = tokenizer.convert_tokens_to_ids("yes")
    token_false = tokenizer.convert_tokens_to_ids("no")
    instruction = "Given a job title, judge whether the skill is relevant to the job."

    def fmt(query_text, doc_text):
        msg = [{"role": "system", "content": "Judge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be \"yes\" or \"no\"."},
               {"role": "user", "content": f"<Instruct>: {instruction}\n<Query>: {query_text}\n<Document>: {doc_text}"}]
        return tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)

    q_ids = list(queries.keys())
    c_ids = list(corpus.keys())
    scores = {}
    for q_id in tqdm(q_ids, desc="QwenRerank"):
        query_text = get_query_text(queries[q_id], descriptions=query_descriptions, q_id=q_id)
        texts = [fmt(query_text, get_corpus_text(corpus, c_id)) for c_id in c_ids]
        all_scores = []
        for i in range(0, len(texts), batch_size):
            inputs = tokenizer(texts[i:i+batch_size], padding=True, truncation=True,
                             max_length=4096, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                logits = model_qwen(**inputs).logits[:, -1, :]
                batch_scores = torch.sigmoid(logits[:, token_true] - logits[:, token_false])
                all_scores.extend(batch_scores.cpu().tolist())
        scores[q_id] = {c_ids[j]: all_scores[j] for j in range(len(c_ids))}

    del model_qwen, tokenizer; torch.cuda.empty_cache()
    return scores

def run_crossencoder_cohere(queries, corpus, model_name=None, query_descriptions=None, top_n=None):
    model_name = model_name or CONFIG["CROSSENC_COHERE"]
    co = cohere.ClientV2()
    c_ids = list(corpus.keys())
    corpus_texts = [get_corpus_text(corpus, c_id) for c_id in c_ids]
    scores = {}
    for q_id in tqdm(queries, desc="Cohere Rerank"):
        query_text = get_query_text(queries[q_id], descriptions=query_descriptions, q_id=q_id)
        response = co.rerank(model=model_name, query=query_text,
                              documents=corpus_texts, top_n=top_n or len(corpus_texts))
        q_scores = {c_id: 0.0 for c_id in c_ids}
        for r in response.results: q_scores[c_ids[r.index]] = r.relevance_score
        scores[q_id] = q_scores
        time.sleep(COHERE_SLEEP)
    return scores

def run_crossencoder_voyage(queries, corpus, model_name=None, query_descriptions=None, top_n=None):
    model_name = model_name or CONFIG["CROSSENC_VOYAGE"]
    vo = voyageai.Client()
    c_ids = list(corpus.keys())
    corpus_texts = [get_corpus_text(corpus, c_id) for c_id in c_ids]
    BATCH = 1000
    scores = {}
    for q_id in tqdm(queries, desc="Voyage Rerank"):
        query_text = get_query_text(queries[q_id], descriptions=query_descriptions, q_id=q_id)
        q_scores = {c_id: 0.0 for c_id in c_ids}
        for cs in range(0, len(corpus_texts), BATCH):
            chunk_texts = corpus_texts[cs:cs+BATCH]
            chunk_cids = c_ids[cs:cs+BATCH]
            response = vo.rerank(model=model_name, query=query_text,
                                  documents=chunk_texts, top_k=len(chunk_texts))
            for r in response.results: q_scores[chunk_cids[r.index]] = r.relevance_score
            time.sleep(VOYAGE_SLEEP)
        scores[q_id] = q_scores
    return scores

print("Cross-encoder functions ready")

In [ ]:
# ============================================================
# Reranking
# ============================================================

def run_rerank(first_stage, queries, corpus, reranker_name, backend,
               top_k=100, query_descriptions=None):
    candidates = {q_id: [c_id for c_id, _ in ranked[:top_k]] for q_id, ranked in first_stage.items()}

    if backend == "local":
        model = CrossEncoder(reranker_name, device=DEVICE, trust_remote_code=True)
        scores = {}
        for q_id in tqdm(candidates, desc="Reranking"):
            qt = get_query_text(queries[q_id], descriptions=query_descriptions, q_id=q_id)
            pairs = [(qt, get_corpus_text(corpus, c_id)) for c_id in candidates[q_id]]
            ps = model.predict(pairs, batch_size=128, show_progress_bar=False)
            scores[q_id] = {candidates[q_id][i]: float(ps[i]) for i in range(len(candidates[q_id]))}
        del model; torch.cuda.empty_cache()

    elif backend == "qwen_reranker":
        tokenizer = AutoTokenizer.from_pretrained(reranker_name, padding_side='left', trust_remote_code=True)
        m = AutoModelForCausalLM.from_pretrained(reranker_name, torch_dtype=torch.float16, device_map=DEVICE, trust_remote_code=True)
        m.eval()
        tt, tf = tokenizer.convert_tokens_to_ids("yes"), tokenizer.convert_tokens_to_ids("no")
        inst = "Given a job title, judge whether the skill is relevant to the job."
        def fmt(qt, dt):
            msg = [{"role":"system","content":"Judge whether the Document meets the requirements. Answer only \"yes\" or \"no\"."},
                   {"role":"user","content":f"<Instruct>: {inst}\n<Query>: {qt}\n<Document>: {dt}"}]
            return tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        scores = {}
        for q_id in tqdm(candidates, desc="Qwen Reranking"):
            qt = get_query_text(queries[q_id], descriptions=query_descriptions, q_id=q_id)
            texts = [fmt(qt, get_corpus_text(corpus, c_id)) for c_id in candidates[q_id]]
            all_s = []
            for i in range(0, len(texts), 16):
                inp = tokenizer(texts[i:i+16], padding=True, truncation=True, max_length=4096, return_tensors="pt").to(DEVICE)
                with torch.no_grad():
                    logits = m(**inp).logits[:, -1, :]
                    all_s.extend(torch.sigmoid(logits[:, tt] - logits[:, tf]).cpu().tolist())
            scores[q_id] = {candidates[q_id][j]: all_s[j] for j in range(len(candidates[q_id]))}
        del m, tokenizer; torch.cuda.empty_cache()

    elif backend == "cohere":
        co = cohere.ClientV2(); scores = {}
        for q_id in tqdm(candidates, desc="Cohere Rerank"):
            qt = get_query_text(queries[q_id], descriptions=query_descriptions, q_id=q_id)
            docs = [get_corpus_text(corpus, c_id) for c_id in candidates[q_id]]
            resp = co.rerank(model=reranker_name, query=qt, documents=docs, top_n=len(docs))
            scores[q_id] = {candidates[q_id][r.index]: r.relevance_score for r in resp.results}
            time.sleep(COHERE_SLEEP)

    elif backend == "voyage":
        vo = voyageai.Client(); scores = {}
        for q_id in tqdm(candidates, desc="Voyage Rerank"):
            qt = get_query_text(queries[q_id], descriptions=query_descriptions, q_id=q_id)
            docs = [get_corpus_text(corpus, c_id) for c_id in candidates[q_id]]
            resp = vo.rerank(model=reranker_name, query=qt, documents=docs, top_k=len(docs))
            scores[q_id] = {candidates[q_id][r.index]: r.relevance_score for r in resp.results}
            time.sleep(VOYAGE_SLEEP)

    return scores

def reciprocal_rank_fusion(run_results_list, k=60):
    all_q = set()
    for results in run_results_list: all_q.update(results.keys())
    fused = {}
    for q_id in all_q:
        s = defaultdict(float)
        for results in run_results_list:
            if q_id in results:
                for rank, (c_id, _) in enumerate(results[q_id]):
                    s[c_id] += 1.0 / (k + rank + 1)
        fused[q_id] = dict(s)
    return fused

# Reranker configs
reranker_configs = [
    ("bge", CONFIG["CROSSENC_BGE"], "local"),
    ("qwen3rnk", CONFIG["CROSSENC_QWEN"], "qwen_reranker"),
    ("voyage", CONFIG["CROSSENC_VOYAGE"], "voyage"),
    ("cohere", CONFIG["CROSSENC_COHERE"], "cohere"),
]

print("Reranking + Fusion ready")

---
## 5. LLM Description Generation (Val + Test)

In [ ]:
# ============================================================
# Description generation functions
# ============================================================
DESCRIPTION_PROMPT = """You are an expert HR assistant specialized in writing job descriptions.
Generate a highly concise, professional, and factual job description based ONLY on the provided job title.
Focus on typical key responsibilities. Crucially, explicitly list 2-3 common, essential skills
associated with the role within the description. Do not add information not directly implied by the
job title. The description must be brief, ideally 2-3 sentences maximum. Output only the description
text, without any preamble or introductory phrases."""

def generate_descriptions_openai(job_titles, model=None):
    model = model or CONFIG["LLM_GPT"]
    client = openai.OpenAI()
    descriptions = {}
    for q_id, title in tqdm(job_titles.items(), desc=f"OpenAI ({model})"):
        try:
            response = client.chat.completions.create(model=model, max_completion_tokens=200, temperature=0.3,
                messages=[{"role": "system", "content": DESCRIPTION_PROMPT},
                          {"role": "user", "content": f"Job title: {title}"}])
            descriptions[q_id] = response.choices[0].message.content.strip()
        except Exception as e:
            print(f"  Error for {q_id}: {e}"); descriptions[q_id] = ""
    return descriptions

def generate_descriptions_anthropic(job_titles, model=None):
    model = model or CONFIG["LLM_OPUS"]
    client = anthropic.Anthropic()
    descriptions = {}
    for q_id, title in tqdm(job_titles.items(), desc=f"Claude ({model})"):
        try:
            response = client.messages.create(model=model, max_tokens=200, system=DESCRIPTION_PROMPT,
                messages=[{"role": "user", "content": f"Job title: {title}"}])
            descriptions[q_id] = response.content[0].text.strip()
        except Exception as e:
            print(f"  Error for {q_id}: {e}"); descriptions[q_id] = ""
    return descriptions

def generate_descriptions_qwen(job_titles, model=None):
    model = model or CONFIG["LLM_QWEN"]
    client = openai.OpenAI(base_url=CONFIG["QWEN_BASE_URL"], api_key=CONFIG["TOGETHER_API_KEY"])
    descriptions = {}
    for q_id, title in tqdm(job_titles.items(), desc=f"Qwen ({model})"):
        try:
            response = client.chat.completions.create(model=model, max_tokens=200, temperature=0.3,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
                messages=[{"role": "system", "content": DESCRIPTION_PROMPT},
                          {"role": "user", "content": f"Job title: {title}"}])
            descriptions[q_id] = response.choices[0].message.content.strip()
        except Exception as e:
            print(f"  Error for {q_id}: {e}"); descriptions[q_id] = ""
    return descriptions

print("Description generation ready")

In [ ]:
# ============================================================
# Generate or load descriptions for BOTH val and test
# ============================================================
desc_cache = {"val": {}, "test": {}}

for split_name, queries in [("val", val_q), ("test", test_q)]:
    for llm_name, gen_fn in [("qwen3", generate_descriptions_qwen),
                              ("opus", generate_descriptions_anthropic),
                              ("gpt54", generate_descriptions_openai)]:
        cache_path = os.path.join(CONFIG["DESC_DIR"], f"{split_name}_desc_{llm_name}.json")

        if os.path.exists(cache_path):
            with open(cache_path) as f:
                desc_cache[split_name][llm_name] = json.load(f)
            print(f"Loaded cached: {split_name}_{llm_name} ({len(desc_cache[split_name][llm_name])} entries)")
        else:
            print(f"Generating: {split_name}_{llm_name}...")
            desc_cache[split_name][llm_name] = gen_fn(queries)
            with open(cache_path, 'w') as f:
                json.dump(desc_cache[split_name][llm_name], f)
            print(f"Saved: {cache_path}")

# The best LLM was Qwen3 from Phase 1 val results
BEST_LLM = "qwen3"
print(f"\nBest LLM: {BEST_LLM}")
for split_name in ["val", "test"]:
    print(f"  {split_name} descriptions: {list(desc_cache[split_name].keys())}")

---
# Phase 1: Run All 39 Experiments on Val and Test

Each experiment runs on both splits. Results are logged to W&B with `val_` / `test_` prefixes.

In [ ]:
# ============================================================
# Master experiment runner
# ============================================================

def run_all_experiments(split_name, queries, corpus, qrels_df, best_descriptions):
    """Run all 39 experiments for a given split."""
    print(f"\n{'#'*80}")
    print(f"# RUNNING ALL EXPERIMENTS ON: {split_name.upper()}")
    print(f"{'#'*80}")

    has_qrels = qrels_df is not None

    # ============================================================
    # GROUP 1: BM25
    # ============================================================
    print(f"\n--- Group 1: BM25 ---")

    # bm25_raw
    scores = run_bm25(queries, corpus)
    bm25_raw_ranked = run_and_log("bm25_raw", "group_1_bm25", scores, split_name, qrels_df,
                                   {"retrieval_method": "bm25", "augmentation_llm": "none"})

    # bm25 augmented (3 LLMs)
    for llm_name in ["qwen3", "opus", "gpt54"]:
        if llm_name in desc_cache[split_name]:
            scores = run_bm25(queries, corpus, query_descriptions=desc_cache[split_name][llm_name])
            run_and_log(f"bm25_aug_{llm_name}", "group_1_bm25", scores, split_name, qrels_df,
                        {"retrieval_method": "bm25", "augmentation_llm": llm_name})

    # ============================================================
    # GROUP 2: Bi-Encoder Raw
    # ============================================================
    print(f"\n--- Group 2: Bi-Encoder Raw ---")

    scores = run_biencoder_local(queries, corpus, model_name=CONFIG["BIENC_QWEN"],
        query_prefix="Instruct: Given a job title, retrieve relevant skills.\nQuery: ")
    bienc_raw_qwen = run_and_log("bienc_raw_qwen3emb", "group_2_bienc_raw", scores, split_name, qrels_df,
                                  {"retrieval_method": "bi-encoder", "model_name": CONFIG["BIENC_QWEN"]})

    scores = run_biencoder_local(queries, corpus, model_name=CONFIG["BIENC_BGE"])
    run_and_log("bienc_raw_bge_m3", "group_2_bienc_raw", scores, split_name, qrels_df,
                {"retrieval_method": "bi-encoder", "model_name": CONFIG["BIENC_BGE"]})

    scores = run_biencoder_openai(queries, corpus)
    run_and_log("bienc_raw_openai", "group_2_bienc_raw", scores, split_name, qrels_df,
                {"retrieval_method": "bi-encoder", "model_name": CONFIG["BIENC_OPENAI"]})

    # ============================================================
    # GROUP 3: Bi-Encoder Augmented
    # ============================================================
    print(f"\n--- Group 3: Bi-Encoder Augmented ---")

    if best_descriptions:
        scores = run_biencoder_local(queries, corpus, model_name=CONFIG["BIENC_QWEN"],
            query_prefix="Instruct: Given a job title and its description, retrieve relevant skills.\nQuery: ",
            query_descriptions=best_descriptions)
        bienc_aug_qwen = run_and_log("bienc_aug_qwen3emb", "group_3_bienc_aug", scores, split_name, qrels_df,
                                      {"retrieval_method": "bi-encoder", "model_name": CONFIG["BIENC_QWEN"],
                                       "augmentation_llm": BEST_LLM})

        scores = run_biencoder_local(queries, corpus, model_name=CONFIG["BIENC_BGE"],
                                      query_descriptions=best_descriptions)
        run_and_log("bienc_aug_bge_m3", "group_3_bienc_aug", scores, split_name, qrels_df,
                    {"retrieval_method": "bi-encoder", "model_name": CONFIG["BIENC_BGE"],
                     "augmentation_llm": BEST_LLM})

        scores = run_biencoder_openai(queries, corpus, query_descriptions=best_descriptions)
        run_and_log("bienc_aug_openai", "group_3_bienc_aug", scores, split_name, qrels_df,
                    {"retrieval_method": "bi-encoder", "model_name": CONFIG["BIENC_OPENAI"],
                     "augmentation_llm": BEST_LLM})

    # ============================================================
    # GROUP 4: Cross-Encoder Raw
    # ============================================================
    print(f"\n--- Group 4: Cross-Encoder Raw ---")

    crossenc_cfgs = [
        ("crossenc_raw_bge", CONFIG["CROSSENC_BGE"], "local"),
        ("crossenc_raw_qwen3rnk", CONFIG["CROSSENC_QWEN"], "qwen_reranker"),
        ("crossenc_raw_voyage", CONFIG["CROSSENC_VOYAGE"], "voyage"),
        ("crossenc_raw_cohere", CONFIG["CROSSENC_COHERE"], "cohere"),
    ]
    for rn, mn, be in crossenc_cfgs:
        try:
            if be == "local": scores = run_crossencoder_local(queries, corpus, mn)
            elif be == "qwen_reranker": scores = run_crossencoder_qwen_reranker(queries, corpus, mn)
            elif be == "cohere": scores = run_crossencoder_cohere(queries, corpus, mn)
            elif be == "voyage": scores = run_crossencoder_voyage(queries, corpus, mn)
            run_and_log(rn, "group_4_crossenc_raw", scores, split_name, qrels_df,
                        {"retrieval_method": "cross-encoder", "model_name": mn})
        except Exception as e:
            print(f"  ERROR ({rn}): {e}")

    # ============================================================
    # GROUP 5: Cross-Encoder Augmented
    # ============================================================
    print(f"\n--- Group 5: Cross-Encoder Augmented ---")

    if best_descriptions:
        for rn, mn, be in crossenc_cfgs:
            rn_aug = rn.replace("_raw_", "_aug_")
            try:
                if be == "local": scores = run_crossencoder_local(queries, corpus, mn, query_descriptions=best_descriptions)
                elif be == "qwen_reranker": scores = run_crossencoder_qwen_reranker(queries, corpus, mn, query_descriptions=best_descriptions)
                elif be == "cohere": scores = run_crossencoder_cohere(queries, corpus, mn, query_descriptions=best_descriptions)
                elif be == "voyage": scores = run_crossencoder_voyage(queries, corpus, mn, query_descriptions=best_descriptions)
                run_and_log(rn_aug, "group_5_crossenc_aug", scores, split_name, qrels_df,
                            {"retrieval_method": "cross-encoder", "model_name": mn, "augmentation_llm": BEST_LLM})
            except Exception as e:
                print(f"  ERROR ({rn_aug}): {e}")

    # ============================================================
    # GROUP 6: Rerank from Bi-Encoder
    # ============================================================
    print(f"\n--- Group 6: Rerank from Bi-Encoder ---")

    for aug_label, first_stage in [("raw", bienc_raw_qwen),
                                     ("aug", bienc_aug_qwen if best_descriptions else None)]:
        if first_stage is None: continue
        descs = best_descriptions if aug_label == "aug" else None
        for rid, mn, be in reranker_configs:
            rn = f"rerank_bienc_{aug_label}_{rid}"
            try:
                scores = run_rerank(first_stage, queries, corpus, mn, be,
                                    top_k=CONFIG["RERANK_TOP_K"], query_descriptions=descs)
                run_and_log(rn, "group_6_rerank_bienc", scores, split_name, qrels_df,
                            {"retrieval_method": "rerank", "reranker": mn, "augmentation_llm": aug_label})
            except Exception as e:
                print(f"  ERROR ({rn}): {e}")

    # ============================================================
    # GROUP 7: Rerank from BM25
    # ============================================================
    print(f"\n--- Group 7: Rerank from BM25 ---")

    # Get best BM25 augmented
    bm25_aug_scores = run_bm25(queries, corpus, query_descriptions=best_descriptions) if best_descriptions else None
    bm25_aug_ranked = results_to_ranked(bm25_aug_scores) if bm25_aug_scores else None

    for aug_label, first_stage in [("raw", bm25_raw_ranked),
                                     ("aug", bm25_aug_ranked)]:
        if first_stage is None: continue
        descs = best_descriptions if aug_label == "aug" else None
        for rid, mn, be in reranker_configs:
            rn = f"rerank_bm25{aug_label}_{rid}"
            try:
                scores = run_rerank(first_stage, queries, corpus, mn, be,
                                    top_k=CONFIG["RERANK_TOP_K"], query_descriptions=descs)
                run_and_log(rn, "group_7_rerank_bm25", scores, split_name, qrels_df,
                            {"retrieval_method": "rerank", "reranker": mn, "augmentation_llm": aug_label})
            except Exception as e:
                print(f"  ERROR ({rn}): {e}")

    # ============================================================
    # GROUP 8: Fusion
    # ============================================================
    print(f"\n--- Group 8: Fusion ---")

    fused_raw = reciprocal_rank_fusion([bm25_raw_ranked, bienc_raw_qwen], k=CONFIG["RRF_K"])
    fusion_raw_ranked = run_and_log("fusion_raw", "group_8_fusion", fused_raw, split_name, qrels_df,
                                     {"retrieval_method": "fusion", "fusion_k": CONFIG["RRF_K"]})

    if bm25_aug_ranked and best_descriptions:
        fused_aug = reciprocal_rank_fusion([bm25_aug_ranked, bienc_aug_qwen], k=CONFIG["RRF_K"])
        fusion_aug_ranked = run_and_log("fusion_aug", "group_8_fusion", fused_aug, split_name, qrels_df,
                                         {"retrieval_method": "fusion", "augmentation_llm": BEST_LLM})

        fused_mixed = reciprocal_rank_fusion([bm25_aug_ranked, bienc_raw_qwen], k=CONFIG["RRF_K"])
        run_and_log("fusion_mixed", "group_8_fusion", fused_mixed, split_name, qrels_df,
                    {"retrieval_method": "fusion", "augmentation_llm": "mixed"})

    # ============================================================
    # GROUP 9: Fusion + Rerank
    # ============================================================
    print(f"\n--- Group 9: Fusion + Rerank ---")

    # Use Voyage (best reranker from Phase 1 val)
    best_rnk = ("voyage", CONFIG["CROSSENC_VOYAGE"], "voyage")
    rid, mn, be = best_rnk

    try:
        scores = run_rerank(fusion_raw_ranked, queries, corpus, mn, be, top_k=CONFIG["RERANK_TOP_K"])
        run_and_log("fusion_rerank_raw", "group_9_fusion_rerank", scores, split_name, qrels_df,
                    {"retrieval_method": "fusion+rerank", "reranker": mn})
    except Exception as e:
        print(f"  ERROR (fusion_rerank_raw): {e}")

    if best_descriptions:
        try:
            scores = run_rerank(fusion_aug_ranked, queries, corpus, mn, be,
                                top_k=CONFIG["RERANK_TOP_K"], query_descriptions=best_descriptions)
            run_and_log("fusion_rerank_aug", "group_9_fusion_rerank", scores, split_name, qrels_df,
                        {"retrieval_method": "fusion+rerank", "reranker": mn, "augmentation_llm": BEST_LLM})
        except Exception as e:
            print(f"  ERROR (fusion_rerank_aug): {e}")

    print(f"\n{'#'*80}")
    print(f"# COMPLETED: {split_name.upper()} — {len(ALL_RESULTS[split_name])} runs logged")
    print(f"{'#'*80}")

print("Master experiment runner ready")

---
## Run on Validation Set

In [ ]:
# ============================================================
# Run all experiments on VAL
# ============================================================
val_best_desc = desc_cache["val"].get(BEST_LLM, {})

run_all_experiments(
    split_name="val",
    queries=val_q,
    corpus=val_c,
    qrels_df=dev_qrels_df,
    best_descriptions=val_best_desc,
)

---
## Run on Test Set

In [ ]:
# ============================================================
# Run all experiments on TEST
# ============================================================
test_best_desc = desc_cache["test"].get(BEST_LLM, {})

run_all_experiments(
    split_name="test",
    queries=test_q,
    corpus=test_c,
    qrels_df=test_qrels_df if has_test_qrels else None,
    best_descriptions=test_best_desc,
)

---
# Results Summary

In [ ]:
# ============================================================
# Display results for both splits
# ============================================================

for split_name in ["val", "test"]:
    if ALL_RESULTS[split_name]:
        df = pd.DataFrame(ALL_RESULTS[split_name])
        df = df.sort_values("map", ascending=False).reset_index(drop=True)

        print(f"\n{'='*90}")
        print(f"  {split_name.upper()} RESULTS — {len(df)} runs (sorted by MAP)")
        print(f"{'='*90}")
        display_cols = ["run_name", "group", "map", "mrr", "ndcg", "precision@5", "precision@10", "precision@100"]
        available = [c for c in display_cols if c in df.columns]
        print(df[available].to_string(index=False, float_format="{:.4f}".format))

        # Save CSV
        csv_path = os.path.join(CONFIG["OUTPUT_DIR"], f"{split_name}_all_results.csv")
        df.to_csv(csv_path, index=False)
        print(f"\nSaved: {csv_path}")

        # Best per group
        print(f"\n  BEST PER GROUP:")
        for group in df["group"].unique():
            best = df[df["group"] == group].iloc[0]
            m = best.get("map", "N/A")
            m_str = f"{m:.4f}" if isinstance(m, float) else str(m)
            print(f"    {group:30s} → {best['run_name']:35s} MAP={m_str}")
    else:
        print(f"\n{split_name.upper()}: No results recorded")

In [ ]:
# ============================================================
# Upload summary tables to W&B
# ============================================================
for split_name in ["val", "test"]:
    if ALL_RESULTS[split_name]:
        df = pd.DataFrame(ALL_RESULTS[split_name])
        df = df.sort_values("map", ascending=False).reset_index(drop=True)

        try:
            run = wandb.init(project=CONFIG["WANDB_PROJECT"],
                             name=f"{split_name}_results_summary", reinit=True)
            wandb.log({f"{split_name}_results_table": wandb.Table(dataframe=df)})
            wandb.finish()
            print(f"Uploaded {split_name} results table to W&B")
        except Exception as e:
            print(f"W&B upload failed for {split_name}: {e}")

In [ ]:
# ============================================================
# Side-by-side comparison: Val vs Test
# ============================================================
if ALL_RESULTS["val"] and ALL_RESULTS["test"]:
    val_df = pd.DataFrame(ALL_RESULTS["val"])[["run_name", "map", "mrr"]].rename(
        columns={"map": "val_map", "mrr": "val_mrr"})
    test_df = pd.DataFrame(ALL_RESULTS["test"])[["run_name", "map", "mrr"]].rename(
        columns={"map": "test_map", "mrr": "test_mrr"})

    merged = val_df.merge(test_df, on="run_name", how="outer")
    merged = merged.sort_values("val_map", ascending=False).reset_index(drop=True)

    print("\n" + "="*80)
    print("  VAL vs TEST — Side by Side")
    print("="*80)
    print(merged.to_string(index=False, float_format="{:.4f}".format))

    csv_path = os.path.join(CONFIG["OUTPUT_DIR"], "val_vs_test_comparison.csv")
    merged.to_csv(csv_path, index=False)
    print(f"\nSaved: {csv_path}")
else:
    print("Need both val and test results for comparison")

---
## Next Steps

These results using the **official evaluation script** (ranx) provide the definitive metrics.
Proceed to fine-tuning experiments in Phase 2.